In [ ]:
# File and directory operations
import os

# Numerical computations and array handling
import numpy as np

# Data manipulation and CSV handling
import pandas as pd

# Plotting and visualization
import matplotlib.pyplot as plt

# Deep learning framework for model building and training
import tensorflow as tf

# Rich display utilities for Jupyter notebooks (e.g., HTML tables, styled output)
from IPython.display import HTML, display

In [ ]:


# Path to the CSV file containing bounding box annotations
#csv_path = 'C:/Users/samya/PyCharmProject/Pneumonia-Detection_dataset/data/stage_2_train_labels.csv'
csv_path=r"C:\Users\samya\PyCharmProject\Pneumonia-Detection\dataset\Patient_dataset_extract.csv"
# Load the full dataset
labels_df = pd.read_csv(csv_path)

# Filter to include only rows where pneumonia is present (Target == 1)
pneumonia_df = labels_df.copy()

#reset index for cleaner downstream processing
pneumonia_df.reset_index(drop=True, inplace=True)

In [ ]:

def plot_score(hist):
    fig, ax = plt.subplots(5, 1, figsize=(10, 20))  # Corrected 'subplot' to 'subplots'
    ax = ax.ravel()

    for i, met in enumerate(['accuracy', 'precision', 'recall', 'AUC', 'loss']):
        ax[i].plot(hist.history[met])
        ax[i].plot(hist.history['val_' + met])
        ax[i].set_title(f'Model {met}')
        ax[i].set_xlabel('Epochs')
        ax[i].set_ylabel(met)
        ax[i].legend(['Train', 'Validation'])
# 
    plt.tight_layout()
    plt.show()


In [ ]:


# Set directory where .npy files are stored
npy_dir = 'npy_data'
test_Y = np.load(os.path.join(npy_dir, 'test_masks.npy'))  # or y_mask if loaded earlier
test_Y = test_Y / 255.0 if test_Y.max() > 1 else test_Y     # Normalize to [0, 1]
test_Y = np.expand_dims(test_Y, axis=-1) if test_Y.ndim == 3 else test_Y  # Add channel dim
test_Y = test_Y.astype('float32')                           # Match model expectations
# Load preprocessed datasets
train_X_rgb = np.load(os.path.join(npy_dir, 'train_X_rgb.npy'))
train_Y     = np.load(os.path.join(npy_dir, 'train_Y.npy'))
test_X_rgb  = np.load(os.path.join(npy_dir, 'test_X_rgb.npy'))
#test_Y      = np.load(os.path.join(npy_dir, 'test_Y.npy'))
y_mask      = np.load(os.path.join(npy_dir, 'train_masks.npy')) 

In [ ]:
y_mask

In [ ]:
train_X_rgb

In [ ]:
test_Y

In [ ]:
print("Total non-zero pixels in masks:", np.count_nonzero(y_mask))

In [ ]:


labels = pd.read_csv(r"C:\Users\samya\PyCharmProject\Pneumonia-Detection_dataset\data\stage_2_train_labels.csv")
count_normal = (labels['Target'] == 0).sum()
count_pneumonia = (labels['Target'] == 1).sum()
train_count = len(train_X_rgb)

classweight = {
    0: (1 / count_normal) * (train_count / 2.0),
    1: (1 / count_pneumonia) * (train_count / 2.0)
}

In [ ]:
classweight


In [ ]:
from tensorflow import keras
from tensorflow.keras.layers import *
from tensorflow.keras import Model


In [ ]:
METRICS = [
    'accuracy',
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall'),
    tf.keras.metrics.AUC(name='AUC'),
    tf.keras.metrics.TruePositives(name='tp'),
    tf.keras.metrics.TrueNegatives(name='tn'),
    tf.keras.metrics.FalsePositives(name='fp'),
    tf.keras.metrics.FalseNegatives(name='fn'),
    tf.keras.metrics.SpecificityAtSensitivity(0.9, name='specificity_at_sens_90'),
    tf.keras.metrics.SensitivityAtSpecificity(0.9, name='sensitivity_at_spec_90'),
]

In [ ]:
import tensorflow as tf

# Flexible Exponential Decay Function
def get_exponential_decay_fn(lr_initial=0.01, decay_steps=20, decay_rate=0.1):
    """
    Returns a learning rate function using exponential decay.
    """
    return lambda epoch: lr_initial * decay_rate ** (epoch / decay_steps)

# Returns a LearningRateScheduler callback with decay configuration
def get_lr_scheduler_cb(lr_initial=0.01, decay_steps=20, decay_rate=0.1):
    return tf.keras.callbacks.LearningRateScheduler(
        schedule=get_exponential_decay_fn(lr_initial, decay_steps, decay_rate),
        verbose=1
    )

# Returns a ModelCheckpoint callback that saves to a unique file
def get_model_checkpoint_cb(model_name):
    return tf.keras.callbacks.ModelCheckpoint(
        filepath=f"{model_name}.keras",
        save_best_only=True,
        monitor="val_loss",
        mode="min",
        verbose=1
    )

# Shared EarlyStopping callback
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    patience=5,
    restore_best_weights=True,
    monitor="val_loss",
    mode="min",
    verbose=1
)

In [ ]:
train_img_path=(r"C:\Users\samya\PyCharmProject\Pneumonia-Detection_dataset\data\stage_2_train_images" )

In [ ]:
import pydicom

# Example image path and ID
img_path = train_img_path[0]
img_id = os.path.splitext(os.path.basename(img_path))[0]

# Load DICOM to get original shape
dcm = pydicom.dcmread(img_path)
orig_shape = dcm.pixel_array.shape  # (height, width)

# Create mask
mask = create_segmentation_mask(
    img_id=img_id,
    boxes_df=bbox_df,  # Make sure this is filtered to Target == 1
    orig_size=orig_shape[::-1],  # Convert to (width, height)
    new_size=(64, 64)
)

In [ ]:
checkpoint_cb = get_model_checkpoint_cb("unet_model")
lr_scheduler_cb = get_lr_scheduler_cb()

In [ ]:
import numpy as np

def create_segmentation_mask(img_id, boxes_df, orig_size=(1024, 1024), new_size=(64, 64)):
    """
    Generate a binary segmentation mask from bounding boxes for a given image ID.

    Parameters:
        img_id (str or int): Identifier for the image.
        boxes_df (pd.DataFrame): DataFrame containing bounding box info with columns:
                                 ['patientId', 'x', 'y', 'width', 'height'].
        orig_size (tuple): Original image size (width, height).
        new_size (tuple): Desired output mask size (width, height).

    Returns:
        np.ndarray: Binary mask of shape `new_size` with 1s inside bounding boxes.
    """
    mask = np.zeros(new_size, dtype=np.uint8)

    # Filter boxes for the given image ID
    boxes = boxes_df[boxes_df['patientId'] == img_id]

    scale_x = new_size[0] / orig_size[0]
    scale_y = new_size[1] / orig_size[1]

    for _, row in boxes.iterrows():
        x1 = int(row['x'] * scale_x)
        y1 = int(row['y'] * scale_y)
        x2 = int((row['x'] + row['width']) * scale_x)
        y2 = int((row['y'] + row['height']) * scale_y)

        # Clip coordinates to mask bounds
        x1, x2 = np.clip([x1, x2], 0, new_size[0])
        y1, y2 = np.clip([y1, y2], 0, new_size[1])

        mask[y1:y2, x1:x2] = 1

    return mask

In [ ]:
train_img_path = (r"C:\Users\samya\PyCharmProject\Pneumonia-Detection_dataset\data\stage_2_train_images")
import os

TRAIN_IMG_DIR = r"C:\Users\samya\PyCharmProject\Pneumonia-Detection_dataset\data\stage_2_train_images"
train_img_paths = [
    os.path.join(TRAIN_IMG_DIR, f)
    for f in os.listdir(TRAIN_IMG_DIR)
    if f.lower().endswith('.dcm')
]

In [ ]:
bbox_df = pd.read_csv(r"C:\Users\samya\PyCharmProject\Pneumonia-Detection_dataset\data\stage_2_train_labels.csv")
bbox_df = bbox_df[bbox_df['Target'] == 1].copy()

In [ ]:
import pydicom

for path in train_img_paths:
    img_id = os.path.splitext(os.path.basename(path))[0]
    dcm = pydicom.dcmread(path)
    orig_shape = dcm.pixel_array.shape  # This is (height, width)

    # Reorder orig_shape to (width, height)
    orig_size = orig_shape[::-1]

    mask = create_segmentation_mask(
        img_id=img_id,
        boxes_df=bbox_df,
        orig_size=orig_size,
        new_size=(64, 64)
    )

    # Now `mask` is your binary segmentation mask for this image
    # e.g., save or visualize it

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(mask, cmap='gray')
plt.title(f"Segmentation Mask for {img_id}")
plt.axis('off')
plt.show()
break  # Just to stop after one image for quick checking

In [ ]:
import numpy as np
import os
from sklearn.model_selection import train_test_split

# --- Configuration ---
npy_dir = 'npy_data'           # Directory containing .npy files
subset_size = 7000             # Limit dataset size (for faster experimentation)


# --- Reduce Dataset (Optional) ---
X = train_X_rgb[:subset_size]
y = y_mask[:subset_size]

# --- Split into Train/Validation ---
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

# --- Inspect Result ---
print("Training set shape:   ", X_train.shape, y_train.shape)
print("Validation set shape: ", X_val.shape, y_val.shape)

In [ ]:
from tensorflow.keras import layers, models, Input

def build_unet(input_shape=(64, 64, 3)):
    inputs = Input(input_shape)

    # --- Encoder ---
    def conv_block(x, filters):
        x = layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
        x = layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
        return x

    def encoder_block(x, filters):
        f = conv_block(x, filters)
        p = layers.MaxPooling2D(pool_size=(2, 2))(f)
        return f, p

    def decoder_block(x, skip, filters):
        x = layers.Conv2DTranspose(filters, kernel_size=2, strides=2, padding='same')(x)
        x = layers.Concatenate()([x, skip])
        return conv_block(x, filters)

    # Downsampling path
    f1, p1 = encoder_block(inputs, 32)
    f2, p2 = encoder_block(p1, 64)
    f3, p3 = encoder_block(p2, 128)

    # Bottleneck
    b = conv_block(p3, 256)

    # Upsampling path
    d1 = decoder_block(b, f3, 128)
    d2 = decoder_block(d1, f2, 64)
    d3 = decoder_block(d2, f1, 32)

    # Output layer
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(d3)

    model = models.Model(inputs, outputs, name='UNet')
    return model

In [ ]:
unet_model = build_unet((64, 64, 3))  # Assuming your U-Net constructor is ready

unet_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=METRICS
)
unet_model.summary()

In [ ]:
dsd

In [ ]:
unet_history = unet_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=128,
    validation_split=0.15,
    #class_weight=classweight,
    callbacks=[checkpoint_cb, early_stopping_cb, lr_scheduler_cb],
    verbose=1
)


In [ ]:


history = unet_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=128,
    validation_split=0.15,
    class_weight=classweight,
    callbacks=[checkpoint_cb, early_stopping_cb, lr_scheduler_cb],
    verbose=1

)

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# Evaluate the trained model on the test dataset and return results as a dictionary
results = unet_model.evaluate(test_X_rgb, test_Y, return_dict=True)

# Display evaluation metrics in a readable format
print("\nEvaluation Results:")
for metric, value in results.items():
    print(f"{metric:>20}: {value:.4f}")

# Plot training and validation metrics (assumes 'plot_score' is a custom function)
plot_score(unet_history)

# Predict probabilities on the test set
pred_probs = unet_model.predict(test_X_rgb)

# Convert predicted probabilities to binary class labels (threshold = 0.5)
pred_labels = (pred_probs > 0.5).astype("int32")

''' Generate and print the confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

# Flatten and convert to integers
test_Y_flat = test_Y.flatten().astype(int)
pred_labels_flat = pred_labels.flatten().astype(int)

# Now compute metrics
print("\nConfusion Matrix:")
print(confusion_matrix(test_Y_flat, pred_labels_flat))

print("\nClassification Report:")
print(classification_report(test_Y_flat, pred_labels_flat, digits=4))


# Print detailed classification report (precision, recall, F1, etc.)
print("\nClassification Report:")
print(classification_report(test_Y, pred_labels, digits=4))'''

In [ ]:
unique, counts = np.unique(test_Y, return_counts=True)
print(dict(zip(unique, counts)))


In [ ]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input  # or VGG16, MobileNet, etc.
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Load the trained model with custom object
model = load_model('mn_cnn_model.h5', custom_objects={'preprocess_input': preprocess_input})

# Load test data
test_X_rgb = np.load('npy_data/test_X_rgb.npy')  # shape: (N, H, W, 3)
test_Y = np.load('npy_data/test_Y.npy')          # shape: (N, H, W, 1) or (N, 1)

# Evaluate model
results = model.evaluate(test_X_rgb, test_Y, return_dict=True)
print("\nEvaluation Results:")
for metric, value in results.items():
    print(f"{metric:>20}: {value:.4f}")

# Predict and post-process
pred_probs = model.predict(test_X_rgb)
pred_labels = (pred_probs > 0.5).astype("int32")

# Flatten if needed
pred_flat = pred_labels.flatten()
true_flat = test_Y.flatten()

# Metrics
print("\nConfusion Matrix:")
print(confusion_matrix(true_flat, pred_flat))

print("\nClassification Report:")
print(classification_report(true_flat, pred_flat, digits=4))


In [ ]:
# Clear the previous version entirely
del test_Y

# Reload or rebuild from source
test_Y = np.stack(list_of_masks, axis=0)
test_Y = np.expand_dims(test_Y, axis=-1)
test_Y = test_Y / 255.0
test_Y = test_Y.astype('float32')

print("New test_Y shape:", test_Y.shape)